# HunyuanOCR Test: LM Studio (OpenAI-Compatible)

This notebook tests **HunyuanOCR** via the LM Studio local server.

### 1. Requirements

Install the OpenAI Python client:

```bash
uv pip install openai pillow
```


In [5]:
import base64
import time
from openai import OpenAI
from PIL import Image
import io

# Point to the local LM Studio server
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


img_path = "ollama_test_table.png"
base64_image = encode_image(img_path)

print("Setup complete. Ready to talk to LM Studio.")

Setup complete. Ready to talk to LM Studio.


### 2. Run Inference: Document Parsing

We are using the **Full Document Parsing** prompt to capture everything: titles, paragraphs, and tables.


In [6]:
print("Requesting full document parsing from HunyuanOCR...")
start_time = time.time()

# This prompt tells the model to extract EVERYTHING (titles + tables)
document_prompt = "提取文档图片中正文的所有信息用markdown格式表示，其中页眉、页脚部分忽略，表格用html格式表达，按照阅读顺序组织进行解析。"

response = client.chat.completions.create(
    model="tencent/HunyuanOCR",
    messages=[
        {
            "role": "system",
            "content": "You are a precise Arabic document analyst. Transcribe all text, including titles like «الجنح» and «المخالفات», and all tables. Ensure Arabic spelling is accurate (e.g., use ح instead of ع in الجنحة).",
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": document_prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                },
            ],
        },
    ],
    max_tokens=4096,
    temperature=0,
)

output_text = response.choices[0].message.content
end_time = time.time()

print(f"\n--- Inference Finished in {end_time - start_time:.2f} seconds ---")
print("\n--- Raw Result ---")
print(output_text)

Requesting full document parsing from HunyuanOCR...

--- Inference Finished in 9.24 seconds ---

--- Raw Result ---
«الجنح»

| النقط الواجب خصمها | الجنعة | الرقم الترتيبي |
|---------------------|---------|----------------|
| ...                 | ...     | 01             |
| ...                 | ...     | ...            |
| 6                   | سياقة مركبة تحت تأثير الكحول أوتحت تأثير المواد المغدرة رفض الخضوع للرائز المشار إليه في المادة 207 أدناه أو للتحققات أو لاختبارات الكشف المنصوص علها في المادين 208 و213 أدناه | 07             |
| ...                 | ...     | ...            |
| 2                   | السائق الذي وجه إليه الأمر بالتوقف وامتنع من تنفيذه أو لم يحترم الأمر بتوقيف المركبة أو رفض سياقة مركبته أو العمل على سياقتها إلى المحجز أو رفض الامتثال للأوامر القانونية الصادرة إليه | 13             |
| ...                 | ...     | ...            |
| ...                 | ...     | 18             |

«المخالفات»

| النقط الواجب خصمها | المخالفات | الرقم الترتيبي |
|-------

### 3. Display Result as Rendered Document

This cell will display the titles and the tables correctly rendered.


In [8]:
from IPython.display import display, HTML
import re

# If the model used Markdown headers like # or ## for the titles,
# we'll wrap them in simple HTML tags for display if needed.
display(HTML(f"<div style='direction: rtl; text-align: right;'>{output_text}</div>"))